### Config

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import torch, os

# Adjust to point to the actual root of your project
PROJECT_ROOT = Path.cwd().parent  # or Path("/absolute/path/to/your/project")
sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize
from omegaconf import OmegaConf

initialize(config_path="../config", version_base="1.3")
cfg = compose(config_name="config")
print(OmegaConf.to_yaml(cfg))


model:
  shortcode: D2-16-A2
  hf_name: deepseek-ai/DeepSeek-V2-Lite
  company: deepseek
  model_family: deepseek-v2
  model_size: 16B-A2B
  it: base
  plot_name: DeepSeek V2 Lite
  color: '#8e9e00'
  apply_chat_template: 'no'



### Loading the model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

load_model = True

model_name = cfg.model.hf_name
tokenizer = AutoTokenizer.from_pretrained(
    model_name, 
    trust_remote_code=True
)
if load_model:
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        device_map='auto', 
        attn_implementation='eager',  
        trust_remote_code=True
    )
    model.generation_config = GenerationConfig.from_pretrained(model_name)
    model.generation_config.pad_token_id = model.generation_config.eos_token_id
    model.eval()

    text = "The goal of life is to"
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model.generate(**inputs.to(model.device), max_new_tokens=10)

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(result)

/home/p84400019/miniconda3/envs/int/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Record activations, save them, and verify them

In [ ]:
from src.activation_recorder import ActivationRecorder, MultiPromptActivations
from IPython.core.debugger import Pdb

load_from_disk = False
save_dir = "../data/activations"
max_new_tokens=cfg.generation.max_new_tokens
prompts = cfg.generation.default_prompts

if not load_from_disk:
    recorder = ActivationRecorder(model, tokenizer)
    activations = recorder.record_prompts(prompts, max_new_tokens=max_new_tokens)
    activations.verify_recorded_activations(prompts=prompts, max_new_tokens=max_new_tokens, tokenizer=tokenizer, diff_q_size=True)
    activations.save(save_dir)

# Load the activations from disk.
file_path = os.path.join(save_dir, "multi_prompt_activations.pkl")
loaded_activations = MultiPromptActivations.load(file_path)

# Optional: verify the loaded activations match the saved ones.
print("Loaded MultiPromptActivations object has:", len(loaded_activations.prompts), "prompts recorded.")

# Check again the activations
loaded_activations.verify_recorded_activations(prompts=prompts, max_new_tokens=max_new_tokens, tokenizer=tokenizer, diff_q_size=True)

print("Final MultiPromptActivations object has:", len(loaded_activations.prompts), "prompts recorded.")

# Extract the first prompt, first step, first layer, first head
prompt_acts = loaded_activations.prompts[0]
step_acts = prompt_acts.steps[0]
layer_acts = step_acts.layers[0]
attn = layer_acts.attention
for head_idx, head_acts in attn.heads.items():
    print(f"Head {head_idx} activations:")
    print(head_acts.query.shape)
    print(head_acts.attention_weights.shape)
    print(head_acts.attention_outputs.shape)
    print(head_acts.projected_outputs.shape)

ConfigAttributeError: Key 'generation' is not in struct
    full_key: generation
    object_type=dict

In [ ]:
step_acts = prompt_acts.steps[4]
for layer_idx, layer_acts in step_acts.layers.items():
    print(f"Layer {layer_idx} activations:")
    moe_layer_acts = step_acts.layers[2].moe
    for expert_id, expert_acts in moe_layer_acts.experts.items():
        print(f"Expert {expert_id} activations:")
        print(repr(expert_acts))
        print("Gate value:", expert_acts.gate_value)
        print("MLP output:", expert_acts.mlp_output)
        print("Expert output:", expert_acts.expert_output)

Layer 0 activations:
Expert 0 activations:
MoEExpertActivations(layer_index=2, expert_index=0, gate_value=torch.Size([]), mlp_output=torch.Size([2048]), expert_output=torch.Size([2048]), is_shared=False, model_info=ModelInformation(model_name=deepseek-ai/DeepSeek-V2-Lite, model_architecture=DeepseekV2ForCausalLM, num_layers=27, num_attention_heads_per_layer=16, total_num_attention_heads=432, attention_implementation=default, hidden_size=2048, head_dim=128, attention_implementation=default))
Gate value: tensor(0.0571)
MLP output: tensor([ 1.3534e-04,  3.2173e-05, -3.3733e-04,  ..., -8.0709e-04,
        -5.2958e-05,  6.5426e-04])
Expert output: tensor([ 1.3534e-04,  3.2173e-05, -3.3733e-04,  ..., -8.0709e-04,
        -5.2958e-05,  6.5426e-04])
Expert 1 activations:
MoEExpertActivations(layer_index=2, expert_index=1, gate_value=None, mlp_output=None, expert_output=None, is_shared=False, model_info=ModelInformation(model_name=deepseek-ai/DeepSeek-V2-Lite, model_architecture=DeepseekV2ForCa